In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision                                 # Specifically for computer vision (CNN)
from torchvision.datasets import CIFAR10           # Using cifer10 dataset

In [2]:
# Tensorloader and Dataloder
from torch.utils.data import DataLoader
import torchvision.transforms as transforms         # Helping the transform the image data.

                                                    # image => scale(0,1) , normalize => (-1, 1) (z-score normalization)

transform = transforms.Compose([                    # chaining multiple data transformation or augmentation steps together into a single, sequential pipeline
    transforms.ToTensor(),                          # pipeline
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

training_data = CIFAR10(
    root="./Data",
    train=True,
    download=False,
    transform=transform
)
testing_data = CIFAR10(
    root="./Data",
    train=False,
    download=False,
    transform=transform
)

### mean = (0.5, 0.5, 0.5): The value subtracted from the R, G, and B channels.
### std = (0.5, 0.5, 0.5): The value each channel is divided by.

In [3]:
# Loader
train_loader = DataLoader(training_data,batch_size=64,shuffle=True)
test_loader = DataLoader(testing_data,batch_size=64)

# Building CNN

In [4]:
# Conve2d : - bcz the kernal only slides on 2 directions.

class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(

            # layer - 1
            nn.Conv2d(in_channels = 3 , out_channels=32 , kernel_size = 3, padding = 1),  # in_channel(rgb) , out(Feture_map), in the starting there is only 3 kernals.
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),
                
            # layer - 2
            nn.Conv2d(in_channels = 32 , out_channels=64 , kernel_size = 3, padding = 1),  # in_channel(rgb) , out(Feture_map), in the starting there is only 3 kernals.
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),
                
            # layer - 3
            nn.Conv2d(in_channels = 64 , out_channels=128 , kernel_size = 3, padding = 1) , # in_channel(rgb) , out(Feture_map), in the starting there is only 3 kernals.
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),
        )


        self.fullyConnected = nn.Sequential(      # FNN (ANN) part
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10),
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)
        x = self.fullyConnected(x)
        return x
        

### 256 feature maps × 16 (4 width x 4 Height )values per feature map = 4096 total values

In [5]:
model = CNN()

In [6]:
criterian = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [7]:
# model training

epochs = 10

for i in range(epochs):
    model.train()
    epoch_loss = 0.0

    for image,labels in train_loader:
        optimizer.zero_grad()
        
        output = model.forward(image)
        loss = criterian(output,labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        
    print(f"Epoch => {i+1}/{epochs} & Avgloss {(epoch_loss/len(train_loader))}")

Epoch => 1/10 & Avgloss 1.336171504862778
Epoch => 2/10 & Avgloss 0.9056910401415033
Epoch => 3/10 & Avgloss 0.7258037053563101
Epoch => 4/10 & Avgloss 0.6034266445642847
Epoch => 5/10 & Avgloss 0.4988138905114225
Epoch => 6/10 & Avgloss 0.398935950225424
Epoch => 7/10 & Avgloss 0.30996963952470313
Epoch => 8/10 & Avgloss 0.24560279030438578
Epoch => 9/10 & Avgloss 0.18261341958085214
Epoch => 10/10 & Avgloss 0.14217555319742703


In [15]:
def compute_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

train_acc = compute_accuracy(model, train_loader)
test_acc = compute_accuracy(model, test_loader)

print(f"Training Accuracy => {train_acc:.2f}%")
print(f"Testing Accuracy  => {test_acc:.2f}%")

Training Accuracy => 96.11%
Testing Accuracy  => 74.90%


### Prediction

```python
_, predicted = torch.max(output, dim=1)

# predicted = tensor([2, 0, 1, 3])  # Predicted labels
# labels    = tensor([2, 1, 1, 3])  # Actual labels

predicted == labels
# tensor([True, False, True, True])
```

In [14]:
# model evaluation
model.eval()
correct_labels = 0
total = 0
with torch.no_grad():
    for image,labels in test_loader:
        outputs = model(image)
        _ , predicted = torch.max(outputs,1)
        correct_labels += (predicted == labels).sum().item()
        total += labels.size(0)

        training_pred = model()
        training_loss = criterian()

print(f"Total => {total}")
print(f"Correct => {correct_labels}")   
print(f"Accuracy => {(correct_labels/total)*100}%")

Total => 10000
Correct => 7490
Accuracy => 74.9%
